# UFC Fight History Scraper (Checkpointed)


## What it does

Scrapes UFCStats fight history into a single CSV for model training, with:

- per-fight checkpointing (SQLite)
- chunked runs (`--max-events`, `--max-fights`)
- safe resume after disconnects/restarts
- leakage-safe pre-fight features and outcome labels

Feature coverage is tuned for fight-outcome modeling:

- Fight context: event/date/location, bout order, weight class, title flag, scheduled rounds.
- Fighter bio: age at fight, height, reach, stance.
- Fighter pre-fight record: wins/losses/draws/no-contests, streak, rest days.
- Method-aware record splits: KO/Sub/Decision wins and losses.
- Pre-fight efficiency (from prior fights only): striking pace/accuracy/defense, takedown pace/accuracy/defense, control time, knockdown and submission rates.
- Matchup deltas: age/height/reach plus wins/losses/streak, win/finish rates, pace/accuracy/defense efficiency, and rest differences.


## Install

```bash
pip install -r requirements-scraper.txt
```


## Run (full history)

```bash
python3 scripts/scrape_ufc_fights.py
```

Default outputs:

- CSV: `data/ufc_fights_rnn.csv`
- Checkpoint DB: `data/checkpoints/ufc_fights_checkpoint.sqlite`


## Run in chunks

Process 20 unprocessed events, then stop:

```bash
python3 scripts/scrape_ufc_fights.py --max-events 20
```

Process 200 new fights, then stop:

```bash
python3 scripts/scrape_ufc_fights.py --max-fights 200
```

Re-running continues from latest stored fight/event automatically.


## Fast mode (throughput)

Skip fight detail pages and checkpoint every 50 fights:

```bash
python3 scripts/scrape_ufc_fights.py \
  --max-events 50 \
  --skip-fight-details \
  --commit-every 50 \
  --sleep-seconds 0.0
```

Tradeoff: this is faster, but `time_format` and `scheduled_rounds` may be partially missing.


## Useful options

```bash
python3 scripts/scrape_ufc_fights.py \
  --start-date 2010-01-01 \
  --end-date 2024-12-31 \
  --sleep-seconds 0.05 \
  --timeout-seconds 30 \
  --max-retries 5 \
  --commit-every 10 \
  --log-level INFO
```

Stop an event immediately on first fight parse error:

```bash
python3 scripts/scrape_ufc_fights.py --stop-on-fight-error
```

Default behavior is more resilient: continue processing other fights in the event.

Export CSV from existing checkpoints only (no scraping):

```bash
python3 scripts/scrape_ufc_fights.py --export-only
```


## Notes

- Data source: UFCStats (`http://ufcstats.com`)
- If the site HTML changes, selectors in `scripts/scrape_ufc_fights.py` may need updates.
- Fighter state is tracked chronologically, so each row only uses information available before that bout.
